In [ ]:
# UPAR27 Track3 v2 | aashish | FIX: usyd-community/vitpose-plus-small + MoE dataset_indexHF_TOKEN = os.environ.get("HF_TOKEN", "hf_YOUR_TOKEN_HERE")  # was hardcoded during Kaggle runs; set HF_TOKEN env/secretHF_REPO="ShivRamSaud/upar-track3-pose"import os, time, jsonos.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

In [ ]:
!pip install -q transformers huggingface_hub timm opencv-python pycocotools
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
from huggingface_hub import login, create_repo
login(token=HF_TOKEN)
create_repo(HF_REPO, exist_ok=True)

In [ ]:
!rm -rf /tmp/t3 && git clone --depth 1 https://github.com/MickaelCormier/UPAR-Challenge-2027-Track3 /tmp/t3
!ls /tmp/t3/examples/task3/vitpose_submission
!ls /kaggle/input 2>&1 | head -20

In [ ]:
# CORRECT checkpoint per official export_model.py: usyd-community/vitpose-plus-small (COCO idx 0)
from transformers import VitPoseForPoseEstimation, VitPoseImageProcessor
import torch
MID='usyd-community/vitpose-plus-small'
proc=VitPoseImageProcessor.from_pretrained(MID)
base=VitPoseForPoseEstimation.from_pretrained(MID)
print('loaded',MID,'labels',base.config.num_labels,'size',proc.size)
class CocoWrapper(torch.nn.Module):
 def __init__(self,m): super().__init__(); self.m=m; self.register_buffer('idx',torch.tensor([0]))
 def forward(self,pv):
  return self.m(pixel_values=pv,dataset_index=self.idx.expand(pv.shape[0])).heatmaps
model=base.cuda().train()
print('trainable',sum(p.numel() for p in model.parameters())/1e6)

In [ ]:
COCO-style PoseDS over train_upar_all.json with basename image index (Market1501+PA100K attached); skips missing/PETA; manual mean/std norm; gaussian heatmaps (ears masked). Kept 20000/42736 REAL. Full cell in local copy.

In [ ]:
from torch.utils.data import DataLoader, random_split
from huggingface_hub import HfApi
HAS_DS = ('ds' in dir() and ds is not None)
if HAS_DS:
 n=len(ds); tr,va=random_split(ds,[int(n*0.95),n-int(n*0.95)])
 loader=DataLoader(tr,batch_size=32,shuffle=True,num_workers=4,pin_memory=True,drop_last=True)
 opt=torch.optim.AdamW(model.parameters(),lr=5e-4)
 scaler=torch.amp.GradScaler('cuda'); api=HfApi(); os.makedirs('/tmp/hf3',exist_ok=True)
 t0=time.time()
 for ep in range(30):
  if time.time()-t0>6.5*3600: break
  model.train()
  for x,y in loader:
   x,y=x.cuda(non_blocking=True),y.cuda(non_blocking=True)
   with torch.amp.autocast('cuda'):
    out=model(pixel_values=x,dataset_index=torch.zeros(x.shape[0],dtype=torch.long,device=x.device)).heatmaps
    loss=((out-y)**2).mean()/2
   scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad()
  print(f'ep {ep} loss {loss.item():.4f}')
  torch.save(model.state_dict(),'/tmp/hf3/pytorch_model.bin')
  open('/tmp/hf3/step.json','w').write(json.dumps({'epoch':ep}))
  api.upload_folder(repo_id=HF_REPO,folder_path='/tmp/hf3',commit_message=f'ep {ep}')
 print('DONE')
else:
 print('SKIP training - no annotations/images. Attach data first.')


In [ ]:
TorchScript export like official export_model.py (wrapper + freeze + preprocess.json) + HF upload. 30 epochs trained (loss 0.0011 flat - gray-fallback era), Codabench 0.2935. Superseded by notebook_large_ran.ipynb (0.8143).